This code is for generating a benchmarking SV callset for HapMap Mix with supporting reads.
The germline SVs in HPRC cell lines are supposed to be mosaic variants in HapMap Mix.
The goal is to locate the "True" mosaic SVs in the HapMap mix where we can find the same signal in HPRC cell lines. If this is the case, we can further benchmark our methylation analysis with the supporting reads in HapMap Mix and HRPC cell lines. 


HapMap Mix:
| Sample | HG00438 | HG002 | HG02257 | HG02486 | HG02622 | HG005 |
|--------|-------------|-------------|-------------|-------------|-------------|-------------|
| Proportion | .5% | 2% | 2% | 2% | 10% | 83.5% |

Input:
- Assembly-polished SVs of 6 HPRC celllines.
- 6 individual Sniffles called HPRC celllines.
  - with supporting reads.


In [25]:
import pandas as pd
import pysam
from src.vcf_to_df import read_vcf_to_df_comp
from src.vcf_to_df import read_vcf_to_df    
from tqdm import tqdm

hprc_sample_list = ["HG002", "HG00438", "HG005", "HG02257", "HG02486", "HG02622"]

In [19]:
assembly_based_sv = pysam.VariantFile("./dataset/assembly_based_sv.vcf.gz")
# assembly_based_sv_df = read_vcf_to_df_comp(assembly_based_sv)

In [12]:
for i in hprc_sample_list:
    sample_based_sv = read_vcf_to_df(pysam.VariantFile(f"./dataset/sniffles_call/{i}.sorted_germline.vcf.gz"))


In [ ]:
# uncomment for re-producing the sv_df

# sv_df = pd.DataFrame(columns=["chr", "location", "id", "sv_len"] + hprc_sample_list)
# for record in tqdm(assembly_based_sv.fetch(), desc="Processing records"):
#     df_record = {"chr": record.chrom, "location": record.pos, "id": record.id, "sv_len": record.info["SVLEN"]}
#     for sample_name in hprc_sample_list:
#         df_record.update({sample_name: record.samples[sample_name]['GT']})
#     sv_df.loc[len(sv_df)] = df_record
# sv_df.to_csv(f"./dataset/assembly_based_sv_df.csv", index=False)


Processing records: 106509it [20:27, 86.78it/s]


In [29]:
sv_df = pd.read_csv(f"./dataset/assembly_based_sv_df.csv")

In [36]:
def is_only_one_not_zero_zero(row, sample_list = hprc_sample_list):
    non_zero_count = sum((row[sample] != "(0, 0)")  for sample in sample_list)
    return non_zero_count == 1
sv_only_in_one_sample = sv_df[sv_df.apply(is_only_one_not_zero_zero, axis=1)]
sv_only_in_one_sample

,chr,location,id,sv_len,HG002,HG00438,HG005,HG02257,HG02486,HG02622
1,chr1,710579,HapMapBench.00000002,344,"(0, 0)","(1, 0)","(0, 0)","(0, 0)","(0, 0)","(0, 0)"
2,chr1,710579,HapMapBench.00000003,336,"(0, 0)","(0, 0)","(0, 0)","(1, 0)","(0, 0)","(0, 0)"
3,chr1,710579,HapMapBench.00000004,320,"(0, 0)","(0, 0)","(0, 0)","(0, 1)","(0, 0)","(0, 0)"
4,chr1,710579,HapMapBench.00000005,319,"(0, 0)","(0, 0)","(0, 0)","(0, 0)","(0, 0)","(1, 0)"
5,chr1,711958,HapMapBench.00000006,-90,"(0, 0)","(0, 0)","(0, 0)","(0, 0)","(0, 0)","(0, 1)"
...,...,...,...,...,...,...,...,...,...,...
106501,chr22,50748509,HapMapBench.00106502,151,"(1, 0)","(0, 0)","(0, 0)","(0, 0)","(0, 0)","(0, 0)"
106503,chr22,50748814,HapMapBench.00106504,203,"(0, 0)","(0, 0)","(0, 0)","(1, 0)","(0, 0)","(0, 0)"
106505,chr22,50748963,HapMapBench.00106506,50,"(0, 0)","(0, 0)","(0, 0)","(1, 0)","(0, 0)","(0, 0)"
106506,chr22,50759786,HapMapBench.00106507,51,"(0, 0)","(0, 0)","(0, 0)","(1, 0)","(0, 0)","(0, 0)"


In [ ]:
location_diff = sv_only_in_one_sample['location'].diff().abs()
# Filter out distinct structural variants (SVs) from a DataFrame
distinct_sv_df = sv_only_in_one_sample[(location_diff > 1000) | (location_diff.isna())]
distinct_sv_df

,chr,location,id,sv_len,HG002,HG00438,HG005,HG02257,HG02486,HG02622
1,chr1,710579,HapMapBench.00000002,344,"(0, 0)","(1, 0)","(0, 0)","(0, 0)","(0, 0)","(0, 0)"
5,chr1,711958,HapMapBench.00000006,-90,"(0, 0)","(0, 0)","(0, 0)","(0, 0)","(0, 0)","(0, 1)"
7,chr1,714355,HapMapBench.00000008,-97,"(0, 0)","(0, 0)","(0, 0)","(0, 0)","(0, 0)","(1, 0)"
8,chr1,724994,HapMapBench.00000009,-2184,"(0, 0)","(0, 0)","(0, 0)","(0, 0)","(0, 0)","(0, 1)"
9,chr1,727299,HapMapBench.00000010,97,"(0, 0)","(0, 0)","(0, 0)","(0, 0)","(0, 0)","(0, 1)"
...,...,...,...,...,...,...,...,...,...,...
106495,chr22,50697556,HapMapBench.00106496,302,"(0, 0)","(0, 0)","(0, 0)","(0, 0)","(0, 0)","(1, 0)"
106496,chr22,50715763,HapMapBench.00106497,98,"(0, 0)","(0, 0)","(0, 0)","(0, 1)","(0, 0)","(0, 0)"
106499,chr22,50748482,HapMapBench.00106500,125,"(0, 0)","(0, 1)","(0, 0)","(0, 0)","(0, 0)","(0, 0)"
106506,chr22,50759786,HapMapBench.00106507,51,"(0, 0)","(0, 0)","(0, 0)","(1, 0)","(0, 0)","(0, 0)"


In [39]:
with open("./dataset/distinct_sv.list", "w") as f:
    for index, row in distinct_sv_df.iterrows():
        f.write(f"{row.id}\n")

In [ ]:
# uncomment to get all the distinct SVs.
#  !bcftools view -Oz -o ./dataset/assembly_based_sv_distinct_heterozygous.vcf.gz -i 'ID=@/stornext/snfs4/next-gen/scratch/Yilei/projects/sv_methylation/SniffMeth/dataset/distinct_sv.list' ./dataset/assembly_based_sv.vcf.gz
# !tabix -p vcf ./dataset/assembly_based_sv_distinct_heterozygous.vcf.gz